<a href="https://colab.research.google.com/github/mariliazago0/fundamentos01/blob/main/notebooks/encontro01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Noções Gerais / Pilares
- decomposição
- abstração
- identificação de padrões
- lógica

Exemplos a partir de coletas de dados:
- Site do MRE - Notas da imprensa
    - Link do site - https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/notas-a-imprensa
    - Entender a estrutura da nossa fonte de informação
    - Realizar a coleta em si
    - Inserir as informações e chamá-las num banco de dados

  - Padrão da paginação das notas de imprensa
      - https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/notas-a-imprensa=0
      - https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/notas-a-imprensa=30


Importação de bibliotexas/programas utilizados neste arquivo


In [6]:
!pip install tinydb

In [7]:
from tinydb import TinyDB, Query

def inserir_no_banco(dados, link_noticia):
  arquivo_banco_dados = "nota_mre.json"
  db = TinyDB(arquivo_banco_dados)
  db.insert(dados)
    #evitar dados repetidos no banco
  Buscar = Query()
  verificar_link = db.contains(Buscar.link == link_noticia)
  if not verificar_link:
    print("Inserindo nova informação no banco")
    db.insert(dados)
  else:
    print("Essa informação não será inserida novamente")

In [5]:
import httpx # responsável pelas requisições web
from bs4 import BeautifulSoup # responsável por realizar o web scraping (coleta de dados)

paginas = ['https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/notas-a-imprensa',
            'https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/notas-a-imprensa?b_start:int=30',
            'https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/notas-a-imprensa?b_start:int=60',
           'https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/notas-a-imprensa?b_start:int=90']

def acessa_pagina (link):
  print(f"Estamos na página: {link}")
  try:
    pag_web = httpx.get(link, timeout=10.0)
    pag_web.raise_for_status()
    bs = BeautifulSoup(pag_web.text, 'html.parser')
    return bs
  except httpx.RequestError as exc:
    print(f"Ocorreu um erro ao acessar a página {link}: {exc}")
    return None

#loop for
for pagina in paginas:
  pagina_inteira = acessa_pagina(pagina)
  if pagina_inteira:
    content_div = pagina_inteira.find("div", attrs={"id": 'content-core'})
    if content_div:
      lista_noticias = content_div.find_all("article")
      for noticia in lista_noticias:
        titulo = None
        link_noticia = None
        num_nota = None
        data = None
        horario = None
        lista_paragrafos = []

        titulo_element = noticia.find(attrs={"class": "tileHeadline"})
        if titulo_element:
          titulo = titulo_element.text.strip()
          print(titulo)
        else:
          print("Título não encontrado")

        link_element = noticia.find('a')
        if link_element and 'href' in link_element.attrs:
          link_noticia = link_element['href']
          print(link_noticia)
        else:
          print("Link não encontrado")

        num_nota_element = noticia.find("span", attrs={"class": "subtitle"})
        if num_nota_element:
          num_nota = num_nota_element.text.strip()
          print(num_nota)
        else:
          print("Número da nota não encontrado")

        data_element = noticia.find("i", attrs={"class": "icon-day"})
        if data_element:
          data = data_element.parent.text.strip()
          print(data)
        else:
          print("Data não encontrada")

        horario_element = noticia.find("i", attrs={"class": "icon-hour"})
        if horario_element:
          horario = horario_element.parent.text.strip()
          print(horario)
        else:
          print("Horário não encontrado")

        # Collect content and insert into database for each 'noticia'
        if link_noticia: # Only try to get content if a link was found
          conteudo_noticia_pagina = acessa_pagina(link_noticia)
          if conteudo_noticia_pagina:
            article_body = conteudo_noticia_pagina.find("div", attrs={"property":"rnews:articleBody"})
            if article_body:
              paragrafos = article_body.find_all("p")
              for paragrafo in paragrafos:
                lista_paragrafos.append(paragrafo.text.strip())
          print(lista_paragrafos)
        else:
          print("Não foi possível coletar o conteúdo da notícia por falta de link.")

        dados = {
            "titulo": titulo,
            "num_nota": num_nota,
            "data": data,
            "horario": horario,
            "link": link_noticia,
            "conteudo": lista_paragrafos
        }
        inserir_no_banco(dados, link_noticia)

    else: # This else correctly belongs to 'if content_div:'
      print(f"Div 'content-core' não encontrada na página: {pagina}")


Estamos na página: https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/notas-a-imprensa
Continuidade dos ataques Israelenses contra o Líbano
https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/continuidade-dos-ataques-israelenses-contra-o-libano
NOTA À IMPRENSA N° 121
08/04/2026
18h05
Estamos na página: https://www.gov.br/mre/pt-br/canais_atendimento/imprensa/notas-a-imprensa/continuidade-dos-ataques-israelenses-contra-o-libano
['O Governo brasileiro condena os mais recentes ataques israelenses, realizados hoje, dia 8, contra o Líbano, os quais visaram extensas áreas e deixaram saldo inicial de 254 mortos e 1.165 feridos, de acordo com dados da Defesa Civil libanesa.', 'A intensificação dessa ofensiva ocorre na sequência do anúncio, na última noite, de cessar-fogo no conflito armado no Oriente Médio e ameaça envolver a região em nova escalada de violência e instabilidade.', 'Ao reafirmar seu compromisso com a soberania e a integridade territo